In [2]:
import torch
import torch.nn.functional as F
import random

# load the dataset
words = open('names.txt', 'r').read().splitlines()
print(len(words), words[:5])

# build the character vocabulary
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
print(itos)

32033 ['emma', 'olivia', 'ava', 'isabella', 'sophia']
{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [3]:
import random

# freeze the split ONCE so every experiment uses the same words
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
words_tr  = words[:n1]
words_dev = words[n1:n2]
words_te  = words[n2:]

def build_dataset(words, block_size):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

# proof that block_size is now a knob
Xtr3, _ = build_dataset(words_tr, block_size=3)
Xtr5, _ = build_dataset(words_tr, block_size=5)
print(Xtr3.shape, Xtr5.shape)

torch.Size([182625, 3]) torch.Size([182625, 5])


In [4]:
def run_experiment(block_size, n_emb, n_hidden=200, max_steps=20000, batch_size=32, seed=2147483647):
    # 1. build data at THIS context size
    Xtr, Ytr   = build_dataset(words_tr,  block_size)
    Xdev, Ydev = build_dataset(words_dev, block_size)

    # 2. init the network for THIS (block_size, n_emb)
    g = torch.Generator().manual_seed(seed)
    C  = torch.randn((27, n_emb), generator=g)
    W1 = torch.randn((block_size * n_emb, n_hidden), generator=g)
    b1 = torch.randn(n_hidden, generator=g)
    W2 = torch.randn((n_hidden, 27), generator=g)
    b2 = torch.randn(27, generator=g)
    parameters = [C, W1, b1, W2, b2]
    for p in parameters:
        p.requires_grad = True

    # 3. train
    for i in range(max_steps):
        ix = torch.randint(0, Xtr.shape[0], (batch_size,))
        Xb, Yb = Xtr[ix], Ytr[ix]
        emb = C[Xb]
        h = torch.tanh(emb.view(emb.shape[0], -1) @ W1 + b1)
        logits = h @ W2 + b2
        loss = F.cross_entropy(logits, Yb)
        for p in parameters:
            p.grad = None
        loss.backward()
        lr = 0.1 if i < max_steps * 0.75 else 0.01
        for p in parameters:
            p.data += -lr * p.grad

    # 4. evaluate full-split losses
    @torch.no_grad()
    def eval_loss(X, Y):
        emb = C[X]
        h = torch.tanh(emb.view(emb.shape[0], -1) @ W1 + b1)
        logits = h @ W2 + b2
        return F.cross_entropy(logits, Y).item()

    return eval_loss(Xtr, Ytr), eval_loss(Xdev, Ydev)

In [5]:
import time

t0 = time.time()
tr, dev = run_experiment(block_size=3, n_emb=10)
dt = time.time() - t0

print(f"train {tr:.4f} | dev {dev:.4f} | {dt:.1f}s for one run")
print(f"~{dt * 9 / 60:.1f} min for a rough 9-run estimate")

train 2.2707 | dev 2.2789 | 27.1s for one run
~4.1 min for a rough 9-run estimate


In [6]:
results = []

for block_size in [3, 5, 8]:
    for n_emb in [10, 30, 100]:
        tr, dev = run_experiment(block_size, n_emb)
        results.append({
            'block_size': block_size,
            'n_emb': n_emb,
            'train': tr,
            'dev': dev,
        })
        print(f"block_size={block_size}  n_emb={n_emb:3d}  "
              f"train={tr:.4f}  dev={dev:.4f}")

block_size=3  n_emb= 10  train=2.2846  dev=2.3004
block_size=3  n_emb= 30  train=2.2501  dev=2.3222
block_size=3  n_emb=100  train=2.2270  dev=2.3162
block_size=5  n_emb= 10  train=2.3379  dev=2.3407
block_size=5  n_emb= 30  train=2.3355  dev=2.3627
block_size=5  n_emb=100  train=2.3543  dev=2.4467
block_size=8  n_emb= 10  train=2.5627  dev=2.5610
block_size=8  n_emb= 30  train=2.5688  dev=2.5926
block_size=8  n_emb=100  train=2.4575  dev=2.5039


In [7]:
# sort by dev loss, best first
results.sort(key=lambda r: r['dev'])

print(f"{'block_size':>10} {'n_emb':>6} {'train':>8} {'dev':>8} {'gap':>7}")
for r in results:
    gap = r['dev'] - r['train']
    print(f"{r['block_size']:>10} {r['n_emb']:>6} "
          f"{r['train']:>8.4f} {r['dev']:>8.4f} {gap:>7.4f}")

best = results[0]
print(f"\nbest dev: block_size={best['block_size']}, "
      f"n_emb={best['n_emb']} → dev {best['dev']:.4f}")

block_size  n_emb    train      dev     gap
         3     10   2.2846   2.3004  0.0158
         3    100   2.2270   2.3162  0.0892
         3     30   2.2501   2.3222  0.0721
         5     10   2.3379   2.3407  0.0028
         5     30   2.3355   2.3627  0.0272
         5    100   2.3543   2.4467  0.0924
         8    100   2.4575   2.5039  0.0464
         8     10   2.5627   2.5610 -0.0017
         8     30   2.5688   2.5926  0.0238

best dev: block_size=3, n_emb=10 → dev 2.3004


In [8]:
run_experiment(best['block_size'], best['n_emb'], max_steps=50000)

(2.2161169052124023, 2.239752769470215)